In [1]:
%cd /workspace/verl/verl
! pwd
import sys, os
# import current pwd to sys.path
sys.path.insert(0, os.getcwd())
sys.path.insert(0, "/workspace/verl/verl/LSTLLM")
from LSTLLM.utils import pre_agent_collate_fn

from verl.trainer.main_ppo import create_rl_dataset, create_rl_sampler
from omegaconf import OmegaConf
from transformers import AutoTokenizer

/workspace/verl/verl
/workspace/verl/verl


/usr/local/lib/python3.10/dist-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-12-14 21:50:47,209	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
/usr/local/lib/python3.10/dist-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [ ]:

# from verl.utils.dataset.rl_dataset import RLHFDataset

cfg = OmegaConf.create({
    "data": {
        "tokenizer": None,
        "use_shm": False,
        "train_files": "/workspace/verl/verl/LSTLLM/data/MemoryAgentBench_from_raw_train.parquet",
        "val_files": "/workspace/verl/verl/LSTLLM/data/MemoryAgentBench_from_raw_test.parquet",
        "train_max_samples": 32,
        "val_max_samples": 32,
        "prompt_key": "prompt",
        "reward_fn_key": "data_source",
        "max_prompt_length": 4096,
        "max_response_length": 4096,
        "train_batch_size": 2,
        "val_batch_size": None,
        "tool_config_path": None,
        "return_raw_input_ids": False,
        "return_raw_chat": False,
        "return_full_prompt": False,
        "shuffle": True,
        "seed": None,
        "dataloader_num_workers": 1,
        "image_patch_size": 14,
        "validation_shuffle": False,
        "filter_overlong_prompts": False,
        "filter_overlong_prompts_workers": 1,
        "truncation": "error",
        "image_key": "images",
        "video_key": "videos",
        "trust_remote_code": False,
        "custom_cls": {
            "path": None,
            "name": None
        },  
        "return_multi_modal_inputs": False,
        "sampler": {
            "class_path": None,
            "class_name": None
        },
        "datagen": {
            "path": None,
            "name": None
        },
        "apply_chat_template_kwargs": {},
    },
})



In [3]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B-Instruct-2507")

In [4]:
train_dataset = create_rl_dataset(
    cfg.data.train_files,
    cfg.data,
    tokenizer,
    None,
    max_samples=cfg.data.get("train_max_samples", -1),
)

train_sampler = create_rl_sampler(cfg.data, train_dataset)

Using dataset class: RLHFDataset
dataset len: 2787
selected 32 random samples out of 2787


In [5]:
from torchdata.stateful_dataloader import StatefulDataLoader

num_workers = cfg.data["dataloader_num_workers"]
train_dataloader = StatefulDataLoader(
    dataset=train_dataset,
    batch_size=cfg.data.get("gen_batch_size", cfg.data.train_batch_size),
    num_workers=num_workers,
    drop_last=True,
    collate_fn=pre_agent_collate_fn,
    sampler=train_sampler,
    )

In [12]:
for batch in train_dataloader:
    for agent_role, prompt in zip(batch['agent_role'], batch['pre_agent_prompt']):
        print(agent_role)
        print(prompt)
        print("-"*25)
    break

answer_gen
[{'content': 'You are an assistant focused on delivering useful, accurate, and context-aware responses by leveraging all available memory and current input.', 'role': 'system'}]
-------------------------
fact_split
[{'content': 'You are a precision-oriented analytical agent. When given input, isolate all distinct factual statements, ensuring each is atomic, explicit, and independently meaningful. Avoid inference unless it is strictly entailed.', 'role': 'system'}]
-------------------------
long_mem
[{'content': 'Continuously maintain a clean and coherent memory base. Merge overlapping entries, resolve conflicts, and discard data that has become obsolete or redundant.', 'role': 'system'}]
-------------------------
short_mem
[{'content': 'Transform raw, low-priority facts into compact summaries that preserve intent and context while minimizing verbosity.', 'role': 'system'}]
-------------------------
answer_gen
[{'content': 'Ensure internal consistency across responses by alig

In [ ]:
from datasets import load_dataset
import pandas as pd

# raw_train_df = pd.read_parquet(cfg.data.train_files)

dataset = load_dataset("parquet", data_files=cfg.data.train_files, split="train")
sample = dataset[0]

In [ ]:
sample['extra_info']

'{"data_source": "Accurate_Retrieval", "sub_source": "ruler_qa1_197K", "pre_agent": {"answer_gen": [{"role": "system", "content": "Generate outputs that directly address the user\'s needs, applying stored knowledge and contextual understanding in a coherent and practical manner."}], "fact_split": [{"role": "system", "content": "Extract factual elements without summarization. Preserve original meaning while ensuring that each fact is isolated, concise, and relevance-scored."}], "long_mem": [{"role": "system", "content": "Your role is to organize and maintain structured memory entries. Prioritize clarity, deduplication, and semantic consistency across stored knowledge."}], "short_mem": [{"role": "system", "content": "Transform raw, low-priority facts into compact summaries that preserve intent and context while minimizing verbosity."}]}, "sample_id": "0", "instance_id": "0-q0", "question_idx": 0, "turn_roles": ["history", "history", "history", "history", "history", "history", "history", 